# 🤖 Agentic AI Travel Agent — Built with LangGraph

**Video:** *What is Agentic AI? Explained Simply* | **Channel:** Prashant Nair (@prashantnairofficial)

---

This notebook demonstrates the **exact use case** from the video — an AI travel agent that doesn't just *suggest* plans, it **actually works** through them autonomously.

### What This Agent Does:
1. **Takes a travel request** in natural language
2. **Reasons and plans** — breaks the request into actionable steps
3. **Uses tools** — searches hotels, checks availability, compares prices
4. **Handles failures** — if a hotel is sold out, it reflects and picks an alternative
5. **Produces a final itinerary** — a complete, bookable travel plan

### The 4 Properties of Agentic AI (demonstrated here):
| Property | How It Shows Up |
|----------|----------------|
| **Autonomy** | Agent decides which tools to call and in what order |
| **Reasoning** | Plans steps, reflects on failures, re-plans |
| **Tool Use** | Calls hotel search, price comparison, booking tools |
| **Memory** | Maintains context (budget, preferences) across all steps |

---

**💡 Want to learn how to build this from scratch?** Subscribe to the channel — we'll do a full deep dive into LangGraph, Agentic Design Patterns, and more.

---

**Author:** Prashant Nair | AI & GenAI Practitioner | Principal Trainer  
**Tech Stack:** LangGraph, LangChain, Azure OpenAI (gpt-4o-mini)

## 📦 Step 1: Install Dependencies

In [ ]:
# Install dependencies (watch for errors — don't skip this output!)
!pip install --quiet langchain==0.3.25 langchain-core==0.3.62 langchain-openai==0.3.18 langgraph==0.3.34 grandalf

# Verify installations
import langchain, langgraph, langchain_openai
print(f"✅ langchain=={langchain.__version__}")
print(f"✅ langgraph=={langgraph.__version__}")
print(f"✅ langchain-openai=={langchain_openai.__version__}")
print("\n⚠️  If you see version mismatches, do Runtime → Restart runtime, then re-run this cell.")

## 🔐 Step 2: Configure Azure OpenAI

Store your credentials in **Colab Secrets** (🔑 icon in left panel):  
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_VERSION`
- `AZURE_OPENAI_DEPLOYMENT` (default: `gpt-4o-mini`)

In [ ]:
from google.colab import userdata
import os

os.environ["AZURE_OPENAI_API_KEY"] = userdata.get("AZURE_OPENAI_API_KEY")
os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get("AZURE_OPENAI_ENDPOINT")
os.environ["AZURE_OPENAI_API_VERSION"] = userdata.get("AZURE_OPENAI_API_VERSION")

DEPLOYMENT_NAME = userdata.get("AZURE_OPENAI_DEPLOYMENT") or "gpt-4o-mini"

print(f"✅ Azure OpenAI configured — Deployment: {DEPLOYMENT_NAME}")

## 🧠 Step 3: Initialize the LLM

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_deployment=DEPLOYMENT_NAME,
    temperature=0.3,
    max_tokens=1024,
)

# Quick test
response = llm.invoke("Say 'Agentic AI is ready!' in one line.")
print(response.content)

## 🔧 Step 4: Define the Agent's Tools

These simulate real-world APIs that an agentic travel assistant would call.  
In production, these would be actual API calls to MakeMyTrip, Booking.com, etc.

**Tools defined:**
- `search_hotels` — Find hotels matching criteria
- `check_availability` — Check if a specific hotel has rooms
- `compare_prices` — Get price comparison across options
- `book_hotel` — Make a reservation

In [ ]:
from langchain_core.tools import tool
from typing import Optional
import json
import random

# ── Simulated Hotel Database ──
HOTEL_DB = {
    "Goa": [
        {"name": "Taj Fort Aguada Resort & Spa", "type": "beach resort", "rating": 4.7,
         "price_per_night": 9500, "available": False,
         "location": "Sinquerim Beach, Candolim"},
        {"name": "Novotel Goa Dona Sylvia Resort", "type": "beach resort", "rating": 4.4,
         "price_per_night": 7200, "available": True,
         "location": "Cavelossim Beach, South Goa"},
        {"name": "The Leela Goa", "type": "beach resort", "rating": 4.6,
         "price_per_night": 8800, "available": True,
         "location": "Mobor Beach, Cavelossim"},
        {"name": "Cidade de Goa", "type": "heritage resort", "rating": 4.3,
         "price_per_night": 6500, "available": True,
         "location": "Vainguinim Beach, Dona Paula"},
        {"name": "Alila Diwa Goa", "type": "luxury resort", "rating": 4.5,
         "price_per_night": 8200, "available": True,
         "location": "Majorda Beach, South Goa"},
    ]
}


@tool
def search_hotels(destination: str, hotel_type: Optional[str] = None) -> str:
    """Search for hotels at a destination. Optionally filter by hotel type (e.g., 'beach resort', 'luxury resort')."""
    hotels = HOTEL_DB.get(destination, [])
    if hotel_type:
        hotels = [h for h in hotels if hotel_type.lower() in h["type"].lower()]
    if not hotels:
        return f"No hotels found in {destination}" + (f" matching type '{hotel_type}'" if hotel_type else "")
    result = []
    for h in hotels:
        result.append(
            f"🏨 {h['name']} | {h['type']} | ⭐ {h['rating']} | "
            f"₹{h['price_per_night']:,}/night | 📍 {h['location']}"
        )
    return f"Found {len(result)} hotels in {destination}:\n" + "\n".join(result)


@tool
def check_availability(hotel_name: str, check_in: str, nights: int) -> str:
    """Check room availability for a specific hotel. Returns availability status and total cost."""
    for dest_hotels in HOTEL_DB.values():
        for h in dest_hotels:
            if h["name"].lower() in hotel_name.lower() or hotel_name.lower() in h["name"].lower():
                if h["available"]:
                    total = h["price_per_night"] * nights
                    return (
                        f"✅ {h['name']} — AVAILABLE\n"
                        f"   Check-in: {check_in} | {nights} nights\n"
                        f"   Rate: ₹{h['price_per_night']:,}/night\n"
                        f"   Total: ₹{total:,} (before taxes)\n"
                        f"   Rating: ⭐ {h['rating']} | 📍 {h['location']}"
                    )
                else:
                    return (
                        f"❌ {h['name']} — SOLD OUT for {check_in} ({nights} nights).\n"
                        f"   This is a popular property. Consider alternative options."
                    )
    return f"❓ Hotel '{hotel_name}' not found in our database."


@tool
def compare_prices(destination: str, nights: int, max_budget: int) -> str:
    """Compare hotel prices at a destination within a budget. Shows total cost for the stay and whether it fits the budget."""
    hotels = HOTEL_DB.get(destination, [])
    if not hotels:
        return f"No hotels found in {destination}"
    result = []
    for h in hotels:
        total = h["price_per_night"] * nights
        status = "✅ Within budget" if total <= max_budget else "⚠️ Over budget"
        avail = "Available" if h["available"] else "SOLD OUT"
        result.append(
            f"{'🟢' if h['available'] and total <= max_budget else '🔴'} {h['name']}\n"
            f"   ₹{h['price_per_night']:,}/night × {nights} = ₹{total:,} | {status} | {avail}"
        )
    result.sort()  # Green options first
    return f"Price comparison for {nights} nights in {destination} (Budget: ₹{max_budget:,}):\n\n" + "\n".join(result)


@tool
def book_hotel(hotel_name: str, check_in: str, nights: int, guest_name: str) -> str:
    """Book a hotel room. Returns confirmation details."""
    for dest_hotels in HOTEL_DB.values():
        for h in dest_hotels:
            if h["name"].lower() in hotel_name.lower() or hotel_name.lower() in h["name"].lower():
                if not h["available"]:
                    return f"❌ BOOKING FAILED: {h['name']} is sold out. Please choose another hotel."
                total = h["price_per_night"] * nights
                conf_id = f"GOA{random.randint(100000, 999999)}"
                return (
                    f"═══════════════════════════════════════\n"
                    f"  ✅ BOOKING CONFIRMED\n"
                    f"═══════════════════════════════════════\n"
                    f"  Confirmation ID: {conf_id}\n"
                    f"  Guest: {guest_name}\n"
                    f"  Hotel: {h['name']}\n"
                    f"  Location: {h['location']}\n"
                    f"  Check-in: {check_in}\n"
                    f"  Duration: {nights} nights\n"
                    f"  Rate: ₹{h['price_per_night']:,}/night\n"
                    f"  Total: ₹{total:,} (before taxes)\n"
                    f"  Rating: ⭐ {h['rating']}\n"
                    f"═══════════════════════════════════════\n"
                    f"  Thank you for booking with AI Travel Agent!"
                )
    return f"❓ Hotel '{hotel_name}' not found."


# Register all tools
tools = [search_hotels, check_availability, compare_prices, book_hotel]

print("✅ 4 Tools registered:")
for t in tools:
    print(f"   🔧 {t.name}: {t.description[:80]}...")

## 🏗️ Step 5: Build the Agentic Graph with LangGraph

This is where it all comes together. We define:
- **State** — what the agent remembers across steps
- **Nodes** — the agent (reasoning) and tool executor
- **Edges** — the agentic loop (reason → act → observe → reason again)

This is the **agentic loop** from the video:  
`Plan → Act → Observe → Reflect → Act Again`

In [ ]:
from langgraph.prebuilt import create_react_agent

# ── System prompt: The agent's persona and instructions ──
SYSTEM_PROMPT = """You are an expert AI Travel Agent. You help users plan and book trips.

YOUR WORKFLOW:
1. Understand the user's request (destination, budget, preferences, dates, duration)
2. Search for hotels matching their criteria
3. Check availability for the best options
4. If the top choice is unavailable, REFLECT and pick the next best option — do NOT stop
5. Compare prices against the budget
6. Book the best available option within budget
7. Present a final summary with confirmation details

IMPORTANT RULES:
- Always check availability BEFORE attempting to book
- If a hotel is sold out, explain why you're moving to the next option
- Stay within the user's budget — never book over budget without asking
- Be conversational but efficient
- Show your reasoning at each step — explain WHY you're making each decision

You demonstrate all 4 properties of Agentic AI:
- AUTONOMY: You decide which tools to use and when
- REASONING: You plan, reflect, and adjust when things go wrong
- TOOL USE: You call search, availability, pricing, and booking tools
- MEMORY: You remember the user's preferences throughout the conversation"""

# ── Create the ReAct Agent ──
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print("✅ Agentic Travel Agent created with LangGraph!")
print(f"   Model: Azure OpenAI ({DEPLOYMENT_NAME})")
print(f"   Tools: {len(tools)} registered")
print(f"   Pattern: ReAct (Reason + Act)")

## 📊 Step 6: Visualize the Agent Graph

This is **the diagram from the video** — the agentic loop visualized.  
Notice how the graph loops back from `tools` to `agent` — that's the **reflect and retry** capability.

**This is what separates an agent from a chatbot.** A chatbot goes `input → output`. An agent goes `input → reason → act → observe → reason → act → ... → output`.

In [ ]:
from IPython.display import Image, display

# Generate and display the agent's graph structure
graph_image = agent.get_graph().draw_png()

display(Image(graph_image))

print("\n🔄 The Agentic Loop:")
print("   __start__ → agent → tools → agent → tools → ... → agent → __end__")
print("\n   The agent keeps looping until the task is COMPLETE.")
print("   Each loop = Reason → Act → Observe → Reflect")

## 🚀 Step 7: Run the Agent — The Exact Scenario from the Video

Here's the request from the video:  
> *"Book me a 3-day trip to Goa, budget ₹25,000, I want a beach resort."*

Watch how the agent:
1. Searches for beach resorts in Goa
2. Picks the highest-rated option (Taj Fort Aguada)
3. Discovers it's **SOLD OUT** ❌
4. **Reflects** and moves to the next best option
5. Checks budget constraints
6. Books the best available option ✅

This is **Agentic AI in action** — autonomy, reasoning, tool use, and memory.

In [ ]:
# ── The exact query from the video ──
user_request = (
    "Book me a 3-day trip to Goa. My budget is 25,000 rupees for the hotel. "
    "I want a beach resort. My name is Prashant Nair. "
    "Check-in date is April 15, 2026. "
    "Pick the highest-rated option that's available and within budget."
)

print("👤 USER REQUEST:")
print(f"   {user_request}")
print("\n" + "═" * 60)
print("🤖 AGENT WORKING...")
print("═" * 60 + "\n")

# ── Stream the agent's execution step by step ──
for step in agent.stream(
    {"messages": [{"role": "user", "content": user_request}]},
    stream_mode="updates",
):
    for node_name, node_output in step.items():
        if node_name == "agent":
            msg = node_output["messages"][-1]
            # Check if agent is calling tools
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"🧠 AGENT REASONING → Calling tool: {tc['name']}")
                    print(f"   Args: {json.dumps(tc['args'], indent=2)}")
                    print()
            elif hasattr(msg, "content") and msg.content:
                print("\n" + "═" * 60)
                print("🤖 AGENT FINAL RESPONSE:")
                print("═" * 60)
                print(msg.content)

        elif node_name == "tools":
            for tool_msg in node_output["messages"]:
                print(f"🔧 TOOL RESULT ({tool_msg.name}):")
                print(f"   {tool_msg.content}")
                print()

## 🔍 Step 8: Trace the Agentic Loop

Let's run it again and **count the loops** — this shows the agent's decision-making trace.
Each iteration = one cycle of **Reason → Act → Observe**.

In [ ]:
# ── Run with full trace ──
result = agent.invoke(
    {"messages": [{"role": "user", "content": user_request}]}
)

# ── Analyze the message trace ──
messages = result["messages"]

print("📊 AGENTIC LOOP TRACE")
print("═" * 60)

tool_calls_count = 0
loop_count = 0

for i, msg in enumerate(messages):
    msg_type = type(msg).__name__

    if msg_type == "HumanMessage":
        print(f"\n  ▶ Step {i+1}: USER INPUT")
        print(f"    {msg.content[:100]}...")

    elif msg_type == "AIMessage":
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            loop_count += 1
            for tc in msg.tool_calls:
                tool_calls_count += 1
                print(f"\n  🧠 Step {i+1}: AGENT REASONS → calls {tc['name']}()")
        elif msg.content:
            print(f"\n  ✅ Step {i+1}: AGENT RESPONDS (final answer)")
            print(f"    {msg.content[:150]}...")

    elif msg_type == "ToolMessage":
        print(f"  🔧 Step {i+1}: TOOL RETURNS result from {msg.name}()")

print("\n" + "═" * 60)
print(f"\n📈 SUMMARY:")
print(f"   Total messages in trace: {len(messages)}")
print(f"   Agentic loops (reason→act): {loop_count}")
print(f"   Tool calls made: {tool_calls_count}")
print(f"\n   💡 A regular chatbot would have done this in 1 step.")
print(f"   💡 This agent took {loop_count} reasoning loops to get the best result.")

## 💬 Step 9: Interactive Mode — Ask Your Own Query

Try different scenarios to see how the agent adapts:
- Change the budget to ₹15,000 and see what happens
- Ask for a heritage resort instead of a beach resort
- Request a 5-night stay and watch the agent handle budget constraints

In [ ]:
# ── Try your own query! ──
# Modify the request below and run the cell

custom_request = (
    "I want to go to Goa for 5 nights starting May 1, 2026. "
    "Budget is only 40,000 rupees for the hotel. "
    "I prefer a luxury resort. My name is Prashant. "
    "Find the best option and book it."
)

print("👤 YOUR REQUEST:")
print(f"   {custom_request}")
print("\n" + "═" * 60 + "\n")

for step in agent.stream(
    {"messages": [{"role": "user", "content": custom_request}]},
    stream_mode="updates",
):
    for node_name, node_output in step.items():
        if node_name == "agent":
            msg = node_output["messages"][-1]
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"🧠 AGENT → {tc['name']}({json.dumps(tc['args'])})")
            elif hasattr(msg, "content") and msg.content:
                print(f"\n🤖 AGENT RESPONSE:\n{msg.content}")
        elif node_name == "tools":
            for tool_msg in node_output["messages"]:
                print(f"🔧 {tool_msg.name} → {tool_msg.content}\n")

## 🎯 Step 10: The 4 Properties — Proof from This Notebook

Let's map what just happened back to the 4 properties of Agentic AI:

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           THE 4 PROPERTIES OF AGENTIC AI                   ║
║           — Demonstrated in This Notebook —                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  1. AUTONOMY                                               ║
║     → The agent DECIDED to search, then check, then book   ║
║     → You gave a goal, not step-by-step instructions       ║
║                                                            ║
║  2. REASONING                                              ║
║     → When Taj Fort Aguada was SOLD OUT, the agent         ║
║       reflected and picked the next best option             ║
║     → It didn't crash or ask you what to do                ║
║                                                            ║
║  3. TOOL USE                                               ║
║     → search_hotels(), check_availability(),               ║
║       compare_prices(), book_hotel()                       ║
║     → The agent chose WHICH tools to call and WHEN         ║
║                                                            ║
║  4. MEMORY                                                 ║
║     → Budget (₹25,000) was remembered across all steps     ║
║     → Guest name, dates, preferences — all maintained      ║
║                                                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  🔑 KEY INSIGHT:                                           ║
║  "A chatbot ANSWERS. An agent WORKS."                      ║
║                                                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  📺 SUBSCRIBE for deep dives into:                         ║
║     → Agentic Design Patterns (Reflection, Supervisor)     ║
║     → Building RAG Pipelines with LangChain                ║
║     → LangGraph from scratch                               ║
║                                                            ║
║  Channel: Prashant Nair (@prashantnairofficial)            ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")

---

## 🔜 What's Next?

This notebook showed you **what** Agentic AI can do. The upcoming videos will show you **how** to build it from scratch:

| Video | Topic | Status |
|-------|-------|--------|
| #1 | What is Agentic AI? (This video) | ✅ Done |
| #2 | Build a RAG Pipeline in 10 Minutes | 🔜 Next |
| #3 | LangChain vs LangGraph — When to Use What | 📋 Planned |
| #4 | Agentic Design Patterns Deep Dive | 📋 Planned |
| #5 | Building a Multi-Agent System | 📋 Planned |

**Subscribe** so you don't miss them: [Prashant Nair on YouTube](https://youtube.com/@prashantnairofficial)

---

*Built by Prashant Nair | AI & GenAI Practitioner | Principal Trainer*  
*Tech: LangGraph + LangChain + Azure OpenAI (gpt-4o-mini)*